In [0]:
%sql
use catalog sagar_cap3_cat1;
use schema silver

Data cleaning for cameras table

In [0]:
from pyspark.sql.functions import col, round, isnull, when

cameras_df = spark.table("sagar_cap3_cat1.ref.cameras")

cameras_df = cameras_df.select(
    col("camera_id").cast("string"),
    col("intersection_id").cast("string"),
    col("corridor_id").cast("string"),
    col("zone").cast("string"),
    col("fault_codes").cast("string"),
    round(col("latitude"), 2).alias("latitude"),
    round(col("longitude"), 2).alias("longitude"),
    round(col("uptime_pct"), 2).alias("uptime_pct"),
    col("maintenance_logs")
)

duplicates = cameras_df.groupBy(cameras_df.columns).count().filter(col("count") > 1)
null_check = cameras_df.filter(isnull(col("camera_id")) | isnull(col("intersection_id")) | 
                               isnull(col("corridor_id")) | isnull(col("zone")) | 
                               isnull(col("fault_codes")) | isnull(col("latitude")) | 
                               isnull(col("longitude")) | isnull(col("uptime_pct")) | 
                               isnull(col("maintenance_logs")))

cameras_df = (cameras_df.withColumn("uptime_pct", when((col("uptime_pct") >= 0) & (col("uptime_pct") <= 100), col("uptime_pct")).otherwise(None))
    .withColumn("latitude", when((col("latitude") >= -90) & (col("latitude") <= 90), col("latitude")).otherwise(None))
    .withColumn("longitude", when((col("longitude") >= -180) & (col("longitude") <= 180), col("longitude")).otherwise(None)))

cameras_df.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.silver.cameras')

In [0]:
display(cameras_df)

camera_id,intersection_id,corridor_id,zone,fault_codes,latitude,longitude,uptime_pct,maintenance_logs
CAM0001,X00598,C008,CENTRAL,FOCUS_DRIFT,28.43,77.13,100.0,null
CAM0002,X00424,C002,NORTHWEST,POWER_FLAP,28.55,77.14,95.65,null
CAM0003,X00592,C006,WEST,LENS_SMUDGE;MAINT,28.48,77.0,97.54,"""[{""""date"""": """"2025-06-10"""""
CAM0004,X00712,C007,NORTH,MAINT,28.75,77.21,100.0,"""[{""""date"""": """"2025-06-30"""""
CAM0005,X00779,C003,SOUTH,null,28.48,77.04,96.02,null
CAM0006,X00225,C010,NORTHWEST,LENS_SMUDGE,28.65,77.39,94.99,null
CAM0007,X00526,C004,NORTH,NET_DROPS,28.39,77.17,100.0,null
CAM0008,X00891,C006,WEST,LENS_SMUDGE,28.41,77.21,99.08,null
CAM0009,X00763,C010,NORTHWEST,LENS_SMUDGE,28.72,77.09,96.67,null
CAM0010,X00464,C004,NORTH,POWER_FLAP;MAINT,28.61,77.12,98.58,"""[{""""date"""": """"2025-07-01"""""


Data cleaning for corridors table

In [0]:
from pyspark.sql.functions import col, round

corridors_df = spark.table("sagar_cap3_cat1.ref.corridors")

corridors_df = corridors_df.select(
    col("corridor_id").cast("string"),
    col("corridor_name").cast("string"),
    col("zone").cast("string"),
    round(col("length_km"), 2).alias("length_km")
)
corridors_df.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.silver.corridors')
display(corridors_df)

corridor_id,corridor_name,zone,length_km
C001,Outer Ring,NORTHWEST,8.4
C002,Inner Ring,NORTHWEST,12.1
C003,Airport Express,SOUTH,9.1
C004,Gurgaon Expwy,NORTH,20.9
C005,Noida Link,CENTRAL,7.9
C006,Tech Park Link,WEST,6.7
C007,Riverfront Blvd,NORTH,10.8
C008,Old City Spine,CENTRAL,19.2
C009,Harbor Express,CENTRAL,10.4
C010,Hillway,NORTHWEST,21.4


Data cleaning for holiday_calendar table

In [0]:
from pyspark.sql.functions import col, round

holiday_calendar = spark.table("sagar_cap3_cat1.ref.holiday_calendar")
holiday_calendar=(holiday_calendar.withColumn('date',col('date').cast('date'))
                                .withColumn('holiday_name',col('holiday_name').cast('string')))
holiday_calendar.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.silver.holiday_calendar')
display(holiday_calendar)

date,holiday_name
2025-09-09,Marathon Day


Data cleaning for intersections table

In [0]:
from pyspark.sql.functions import col, round

intersections = spark.table("sagar_cap3_cat1.ref.intersections")

intersections = intersections.select(
    col("intersection_id").cast("string"),
    col("name").cast("string"),
    col("corridor_id").cast("string"),
    col("corridor_name").cast("string"),
    col("zone").cast("string"),
    round(col("latitude"), 2).alias("latitude"),
    round(col("longitude"), 2).alias("longitude"),
    col("school_distance_m").cast("int"),
    col("hospital_distance_m").cast("int"),
    col("near_school").cast("boolean"),
    col("near_hospital").cast("boolean"),
    col("is_top50").cast("boolean")
)
intersections.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.silver.intersections')
display(intersections)

intersection_id,name,corridor_id,corridor_name,zone,latitude,longitude,school_distance_m,hospital_distance_m,near_school,near_hospital,is_top50
X00001,Junction 1,C007,Riverfront Blvd,NORTH,28.46,77.24,null,213,false,true,true
X00002,Junction 2,C007,Riverfront Blvd,NORTH,28.52,77.03,null,null,false,false,false
X00003,Junction 3,C002,Inner Ring,NORTHWEST,28.54,77.13,null,null,false,false,false
X00004,Junction 4,C001,Outer Ring,NORTHWEST,28.71,77.22,null,null,false,false,false
X00005,Junction 5,C009,Harbor Express,CENTRAL,28.5,77.26,null,null,false,false,false
X00006,Junction 6,C004,Gurgaon Expwy,NORTH,28.7,76.97,null,null,false,false,false
X00007,Junction 7,C002,Inner Ring,NORTHWEST,28.78,77.38,null,null,false,false,true
X00008,Junction 8,C006,Tech Park Link,WEST,28.43,77.13,null,null,false,false,false
X00009,Junction 9,C002,Inner Ring,NORTHWEST,28.65,77.04,null,null,false,false,false
X00010,Junction 10,C007,Riverfront Blvd,NORTH,28.48,77.41,null,null,false,false,false


Data cleaning for sensors table

In [0]:
from pyspark.sql.functions import col, round, to_date

sensors = spark.table("sagar_cap3_cat1.ref.sensors")

sensors = sensors.select(
    col("sensor_id").cast("string"),
    col("intersection_id").cast("string"),
    col("corridor_id").cast("string"),
    col("zone").cast("string"),
    col("direction").cast("string"),
    col("model").cast("string"),
    col("firmware").cast("string"),
    col("lane_count").cast("int"),
    round(col("latitude"), 2).alias("latitude"),
    round(col("longitude"), 2).alias("longitude"),
    to_date(col("install_date"), "yyyy-MM-dd").alias("install_date")
)
sensors.write.mode('overwrite').saveAsTable('sagar_cap3_cat1.silver.sensors')
display(sensors)

sensor_id,intersection_id,corridor_id,zone,direction,lane_count,latitude,longitude,model,firmware,install_date
S00001,X00220,C009,CENTRAL,EAST,2,28.837706999999998,77.359922,BX-310,v3.8.8,2024-05-26
S00002,X00585,C010,NORTHWEST,SOUTH,2,28.401315,77.197862,AX-210,v2.5.9,2023-08-09
S00003,X00256,C003,SOUTH,EAST,4,28.824932,77.12070600000001,AX-210,v3.1.5,2022-08-27
S00004,X00812,C002,NORTHWEST,SOUTH,5,28.751991999999998,77.428658,AX-200,v3.4.0,2023-05-10
S00005,X00749,C009,CENTRAL,WEST,2,28.421675999999998,77.042671,BX-310,v2.6.6,2024-04-07
S00006,X00889,C002,NORTHWEST,SOUTH,6,28.679831,77.219692,CX-500,v1.4.8,2022-05-04
S00007,X00043,C009,CENTRAL,SOUTH,4,28.380483,77.10635500000001,AX-200,v3.6.7,2022-11-23
S00008,X00771,C010,NORTHWEST,NORTH,3,28.676559,77.36819299999999,AX-210,v1.5.1,2023-09-02
S00009,X00488,C010,NORTHWEST,NORTH,6,28.694406,77.37986,CX-500,v3.2.2,2023-08-09
S00010,X00493,C003,SOUTH,SOUTH,5,28.700546,77.016459,AX-210,v1.1.4,2023-07-27


Data cleaning for stream_till_now_table

In [0]:
from pyspark.sql.functions import col, round, to_timestamp, mean, struct, when

stream_till_now = spark.table("sagar_cap3_cat1.streamin.stream_till_now")

aqi_mean = stream_till_now.select(mean("aqi")).collect()[0][0]
occupancy_mean = stream_till_now.select(mean("occupancy")).collect()[0][0]
battery_level_mean = stream_till_now.select(mean("battery_level")).collect()[0][0]

processed_df = stream_till_now.select(
    col("sensor_id").cast("string"),
    col("intersection_id").cast("string"),
    col("zone").cast("string"),
    col("corridor_id").cast("string"),
    when(col("lane_status").isNull(), "unknown").otherwise(col("lane_status")).cast("string").alias("lane_status"),
    when(col("speed_unit").isNull(), "km/h").otherwise(col("speed_unit")).cast("string").alias("speed_unit"),
    col("vehicle_count").cast("int"),
    when(col("aqi").isNull(), aqi_mean).otherwise(col("aqi")).cast("int").alias("aqi"),
    round(when(col("occupancy").isNull(), occupancy_mean).otherwise(col("occupancy")), 2).alias("occupancy"),
    round(when(col("battery_level").isNull(), battery_level_mean).otherwise(col("battery_level")), 2).alias("battery_level"),
    round(col("avg_speed_kmh"), 2).alias("avg_speed_kmh"),
    to_timestamp(col("event_ts")).alias("event_ts"),
    to_timestamp(col("ingest_ts")).alias("ingest_ts"),
    struct(
        round(col("geo").getItem("lat"), 2).alias("lat"),
        round(col("geo").getItem("lon"), 2).alias("lon")
    ).alias("geo")
)

processed_df = processed_df.dropDuplicates()

display(processed_df)

processed_df.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.silver.events_data")

sensor_id,intersection_id,zone,corridor_id,lane_status,speed_unit,vehicle_count,aqi,occupancy,battery_level,avg_speed_kmh,event_ts,ingest_ts,geo
S00024,X00139,NORTHWEST,C002,OPEN,km/h,4,38,0.62,0.86,62.45,2025-08-29T01:00:30.000Z,2025-08-29T01:00:33.000Z,"List(28.38, 77.35)"
S00206,X00861,NORTH,C004,OPEN,km/h,4,37,0.62,0.86,61.69,2025-08-29T01:00:30.000Z,2025-08-29T01:00:45.000Z,"List(28.43, 77.32)"
S00193,X00502,NORTHWEST,C001,OPEN,km/h,4,42,0.62,0.86,75.53,2025-08-29T01:01:30.000Z,2025-08-29T01:03:42.000Z,"List(28.41, 77.38)"
S00066,X00089,CENTRAL,C009,OPEN,km/h,5,35,0.62,0.86,63.84,2025-08-29T01:02:00.000Z,2025-08-29T01:02:10.000Z,"List(28.59, 77.07)"
S00207,X00735,CENTRAL,C009,OPEN,km/h,3,39,0.62,0.86,69.75,2025-08-29T01:04:00.000Z,2025-08-29T01:04:22.000Z,"List(28.35, 76.98)"
S00182,X00687,CENTRAL,C008,OPEN,km/h,1,42,0.62,0.86,68.88,2025-08-29T01:07:30.000Z,2025-08-29T01:07:36.000Z,"List(28.78, 77.39)"
S00370,X00623,SOUTH,C003,OPEN,km/h,3,46,0.62,0.86,47.43,2025-08-29T01:07:30.000Z,2025-08-29T01:07:47.000Z,"List(28.38, 77.3)"
S00086,X00722,NORTHWEST,C002,OPEN,km/h,3,50,0.62,0.86,51.83,2025-08-29T01:08:00.000Z,2025-08-29T01:08:03.000Z,"List(28.54, 77.16)"
S00227,X00756,NORTH,C004,OPEN,km/h,0,38,0.62,0.86,62.9,2025-08-29T01:00:00.000Z,2025-08-29T01:00:25.000Z,"List(28.71, 77.27)"
S00425,X00827,NORTHWEST,C001,OPEN,km/h,3,41,0.62,0.86,56.41,2025-08-29T01:00:30.000Z,2025-08-29T01:00:47.000Z,"List(28.6, 77.15)"


In [0]:
from pyspark.sql.functions import col, to_timestamp

weather_all = spark.table("sagar_cap3_cat1.weather.weather_all_zones")

weather_all = weather_all.select(
    col("timestamp").cast("timestamp").alias("timestamp"),
    col("zone").cast("string").alias("zone"),
    col("precip_mm").cast("float").alias("precip_mm"),
    col("temperature_c").cast("float").alias("temperature_c")
)

weather_all.write.mode("overwrite").saveAsTable("sagar_cap3_cat1.silver.weather_all_zones")
display(weather_all)


timestamp,zone,precip_mm,temperature_c
2025-08-29T00:00:00.000Z,NORTHWEST,4.31,25.2
2025-08-29T01:00:00.000Z,NORTHWEST,0.0,26.1
2025-08-29T02:00:00.000Z,NORTHWEST,0.0,26.8
2025-08-29T03:00:00.000Z,NORTHWEST,4.23,25.2
2025-08-29T04:00:00.000Z,NORTHWEST,1.49,22.8
2025-08-29T05:00:00.000Z,NORTHWEST,0.0,21.6
2025-08-29T06:00:00.000Z,NORTHWEST,3.15,22.1
2025-08-29T07:00:00.000Z,NORTHWEST,4.99,26.1
2025-08-29T08:00:00.000Z,NORTHWEST,3.92,26.0
2025-08-29T09:00:00.000Z,NORTHWEST,2.13,26.1
